# **Data Science 112 Final Project: Data Analysis**



# Data Analysis


## Sentiment Analysis

In [ ]:
import re
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px

from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
PROJECT_DIR = Path("/content/drive/MyDrive/DATASCI 112: Final Project - Movie Emotions by Genre")
OUTPUT_DIR = PROJECT_DIR / "outputs"
FIGURE_DIR = OUTPUT_DIR / "figures"
FIGURE_DIR.mkdir(exist_ok=True)

DATA_PATH = OUTPUT_DIR / "scene_level_dialogue_with_api_metadata.csv"

df = pd.read_csv(DATA_PATH)

df = df[df["title"].notna()].copy()

df["dialogue_text"] = df["dialogue_text"].fillna("")

df.head()

,movie_id,scene_number,scene_heading,relative_position,dialogue_text,query_used,tmdb_id,title,runtime,genre,release_date,vote_average,vote_count,popularity,title_score,final_score
0,10thingsihateaboutyou,1,INT. GIRLS' ROOM - DAY,0.011236,Did you change your hair? No. You might wanna ...,10 things i hate about you,4951.0,10 Things I Hate About You,97.0,"Comedy, Romance, Drama",1999-03-30,7.592,8904.0,16.1927,100.0,100.809635
1,10thingsihateaboutyou,2,INT. HALLWAY - DAY- CONTINUOUS,0.022472,We've got your basic beautiful people. Unless ...,10 things i hate about you,4951.0,10 Things I Hate About You,97.0,"Comedy, Romance, Drama",1999-03-30,7.592,8904.0,16.1927,100.0,100.809635
2,10thingsihateaboutyou,3,EXT. SCHOOL COURTYARD - DAY,0.033708,And these delusionals are the White Rastae. Se...,10 things i hate about you,4951.0,10 Things I Hate About You,97.0,"Comedy, Romance, Drama",1999-03-30,7.592,8904.0,16.1927,100.0,100.809635
3,10thingsihateaboutyou,4,INT. CAFETERIA - DAY - CONTINUOUS,0.044944,"Future MBAs- We're all Ivy League, already ac...",10 things i hate about you,4951.0,10 Things I Hate About You,97.0,"Comedy, Romance, Drama",1999-03-30,7.592,8904.0,16.1927,100.0,100.809635
4,10thingsihateaboutyou,5,INT. GUIDANCE COUNSELOR'S OFFICE - DAY,0.056180,"Katarina Stratford. My, my. You've been terr...",10 things i hate about you,4951.0,10 Things I Hate About You,97.0,"Comedy, Romance, Drama",1999-03-30,7.592,8904.0,16.1927,100.0,100.809635


In [ ]:
def assign_story_zone(relative_position):
    if relative_position <= 0.15:
        return "Opening"
    elif relative_position <= 0.30:
        return "Inciting conflict"
    elif relative_position <= 0.65:
        return "Escalation"
    elif relative_position <= 0.90:
        return "Climax zone"
    else:
        return "Resolution"


zone_order = [
    "Opening",
    "Inciting conflict",
    "Escalation",
    "Climax zone",
    "Resolution"
]

df["story_zone"] = df["relative_position"].apply(assign_story_zone)

df["story_zone"] = pd.Categorical(
    df["story_zone"],
    categories=zone_order,
    ordered=True
)

In [ ]:
def get_primary_genre(genre):
    if not isinstance(genre, str) or genre.strip() == "":
        return np.nan

    genre = genre.replace("[", "").replace("]", "")
    genre = genre.replace("'", "").replace('"', "")

    parts = re.split(r",|\|", genre)
    parts = [part.strip() for part in parts if part.strip()]

    if len(parts) > 0:
        return parts[0]
    else:
        return np.nan


df["primary_genre"] = df["genre"].apply(get_primary_genre)

df = df[df["primary_genre"].notna()].copy()

In [ ]:
!pip -q install vaderSentiment

from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

analyzer = SentimentIntensityAnalyzer()

def get_sentiment(text):
    if not isinstance(text, str) or text.strip() == "":
        return np.nan

    return analyzer.polarity_scores(text)["compound"]


df["sentiment_score"] = df["dialogue_text"].apply(get_sentiment)

In [ ]:
def summarize_movie(group):
    zone_sentiments = (
        group
        .groupby("story_zone", observed=False)["sentiment_score"]
        .mean()
        .to_dict()
    )

    return pd.Series({
        "title": group["title"].iloc[0],
        "primary_genre": group["primary_genre"].iloc[0],
        "opening_sentiment": zone_sentiments.get("Opening", np.nan),
        "resolution_sentiment": zone_sentiments.get("Resolution", np.nan)
    })


movie_df = (
    df
    .groupby("movie_id")
    .apply(summarize_movie)
    .reset_index()
)

/tmp/ipykernel_18366/2721786473.py:25: DeprecationWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.



In [ ]:
genre_arc_df = (
    df
    .groupby(["primary_genre", "story_zone"], observed=False, as_index=False)
    .agg(
        avg_sentiment=("sentiment_score", "mean"),
        num_movies=("movie_id", "nunique")
    )
)

In [ ]:
genre_counts = (
    df
    .groupby("primary_genre")["movie_id"]
    .nunique()
    .sort_values(ascending=False)
)

valid_genres = genre_counts[genre_counts >= 5].index.tolist()

## TF-IDF Analysis



In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [ ]:
SENTIMENT_COL = "sentiment_score"

tfidf_sentiment_df = df.copy()

tfidf_sentiment_df["dialogue_text"] = tfidf_sentiment_df["dialogue_text"].fillna("")

Rows: 73160
Movies: 591


In [ ]:
tfidf_filter = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    max_df=0.85,
    min_df=5,
    max_features=4000,
    ngram_range=(1, 1)
)

tfidf_filter.fit(tfidf_sentiment_df["dialogue_text"])

important_words = set(tfidf_filter.get_feature_names_out())

Number of important TF-IDF words: 4000


In [ ]:
def keep_tfidf_words(text):
    words = re.findall(r"[a-zA-Z']+", str(text).lower())

    kept_words = [word for word in words if word in important_words]

    return " ".join(kept_words)


tfidf_sentiment_df["tfidf_filtered_dialogue"] = tfidf_sentiment_df["dialogue_text"].apply(keep_tfidf_words)

,dialogue_text,tfidf_filtered_dialogue,original_word_count,filtered_word_count,percent_words_kept
0,Did you change your hair? No. You might wanna ...,did change hair wanna think leave room enter h...,321,120,0.373832
1,We've got your basic beautiful people. Unless ...,got beautiful people unless talk bother beauti...,85,29,0.341176
2,And these delusionals are the White Rastae. Se...,white white boys lounge grass cloud pot smoke ...,91,35,0.384615
3,"Future MBAs- We're all Ivy League, already ac...",future league someday guys points table variou...,357,109,0.305322
4,"Katarina Stratford. My, my. You've been terr...",ms opinion action yes expression year events q...,134,38,0.283582


In [ ]:
tfidf_sentiment_df["tfidf_filtered_sentiment"] = (
    tfidf_sentiment_df["tfidf_filtered_dialogue"].apply(get_sentiment)
)

tfidf_sentiment_df[[
    "movie_id",
    "scene_number",
    SENTIMENT_COL,
    "tfidf_filtered_sentiment"
]].head()

,movie_id,scene_number,sentiment_score,tfidf_filtered_sentiment
0,10thingsihateaboutyou,1,0.9829,0.9635
1,10thingsihateaboutyou,2,0.6117,0.8225
2,10thingsihateaboutyou,3,0.8875,0.8481
3,10thingsihateaboutyou,4,-0.7730,0.8085
4,10thingsihateaboutyou,5,0.9170,0.9136


In [ ]:
sentiment_compare_df = tfidf_sentiment_df.dropna(
    subset=[SENTIMENT_COL, "tfidf_filtered_sentiment"]
).copy()

sentiment_correlation = sentiment_compare_df[SENTIMENT_COL].corr(
    sentiment_compare_df["tfidf_filtered_sentiment"]
)

Correlation between original and TF-IDF-filtered sentiment: 0.7930778170651979


In [ ]:
tfidf_genre_arc_df = (
    tfidf_sentiment_df
    .dropna(subset=["primary_genre", "tfidf_filtered_sentiment"])
    .groupby(["primary_genre", "story_zone"], observed=False, as_index=False)
    .agg(
        avg_original_sentiment=(SENTIMENT_COL, "mean"),
        avg_tfidf_filtered_sentiment=("tfidf_filtered_sentiment", "mean"),
    )
)

,primary_genre,story_zone,avg_original_sentiment,avg_tfidf_filtered_sentiment,num_scenes,num_movies
0,Action,Opening,0.045403,0.105521,1148,81
1,Action,Inciting conflict,0.029916,0.080573,1190,81
2,Action,Escalation,-0.053467,0.008078,2712,87
3,Action,Climax zone,-0.084048,-0.019706,1890,84
4,Action,Resolution,-0.069517,-0.027324,785,92


In [ ]:
tfidf_genre_arc_df["filtered_minus_original"] = (
    tfidf_genre_arc_df["avg_tfidf_filtered_sentiment"]
    - tfidf_genre_arc_df["avg_original_sentiment"]
)

plot_diff_df = tfidf_genre_arc_df[
    tfidf_genre_arc_df["primary_genre"].isin(valid_genres)
].copy()

fig_tfidf_diff = px.bar(
    plot_diff_df,
    x="story_zone",
    y="filtered_minus_original",
    color="primary_genre",
    barmode="group",
    category_orders={"story_zone": zone_order},
    title="How TF-IDF Filtering Changes Measured Sentiment",
    labels={
        "story_zone": "Story Zone",
        "filtered_minus_original": "Filtered Sentiment - Original Sentiment",
        "primary_genre": "Genre"
    }
)

fig_tfidf_diff.add_hline(
    y=0,
    line_dash="dash",
    line_color="gray"
)

fig_tfidf_diff.update_layout(
    template="plotly_white",
    width=1200,
    height=700
)

fig_tfidf_diff.show()

In [ ]:
plot_A_df = tfidf_sentiment_df.dropna(
    subset=[SENTIMENT_COL, "tfidf_filtered_sentiment"]
).copy()

r = plot_A_df[SENTIMENT_COL].corr(plot_A_df["tfidf_filtered_sentiment"])

plot_A_df = plot_A_df.sample(n=min(2500, len(plot_A_df)), random_state=42)

fig_A = px.scatter(
    plot_A_df,
    x=SENTIMENT_COL,
    y="tfidf_filtered_sentiment",
    color="primary_genre",
    trendline="ols",
    opacity=0.4,
    title=f"Original vs. TF-IDF-Filtered Sentiment (r = {r:.2f})",
    labels={
        SENTIMENT_COL: "Original Scene Sentiment",
        "tfidf_filtered_sentiment": "TF-IDF-Filtered Scene Sentiment",
        "primary_genre": "Genre"
    }
)

fig_A.add_hline(y=0, line_dash="dash", line_color="gray")
fig_A.add_vline(x=0, line_dash="dash", line_color="gray")
fig_A.update_layout(template="plotly_white", width=950, height=600)

fig_A.show()

In [1]:
top_genres = valid_genres[:6]

plot_C_df = tfidf_genre_arc_df[
    tfidf_genre_arc_df["primary_genre"].isin(top_genres)
].copy()

plot_C_df["story_zone"] = pd.Categorical(
    plot_C_df["story_zone"],
    categories=zone_order,
    ordered=True
)

plot_C_df = plot_C_df.sort_values(["primary_genre", "story_zone"])

fig_C = px.line(
    plot_C_df,
    x="story_zone",
    y="avg_tfidf_filtered_sentiment",
    color="primary_genre",
    markers=True,
    title="TF-IDF-Filtered Emotional Arcs by Genre",
    labels={
        "story_zone": "Story Zone",
        "avg_tfidf_filtered_sentiment": "Average TF-IDF-Filtered Sentiment",
        "primary_genre": "Genre"
    }
)

fig_C.add_hline(y=0, line_dash="dash", line_color="gray")
fig_C.update_traces(line=dict(width=4), marker=dict(size=10))
fig_C.update_layout(template="plotly_white", width=950, height=600)

fig_C.show()

NameError: name 'valid_genres' is not defined

## Visualizations


In [ ]:
genre_count_df = (
    df
    .groupby("primary_genre", as_index=False)
    .agg(num_movies=("movie_id", "nunique"))
    .sort_values("num_movies", ascending=False)
)

fig_genre_count = px.bar(
    genre_count_df.head(15),
    x="primary_genre",
    y="num_movies",
    title="Unique Movies Per Genre",
    labels={
        "primary_genre": "Genre",
        "num_movies": "Unique Movies"
    }
)

fig_genre_count.update_layout(
    template="plotly_white",
    xaxis_tickangle=-45
)

fig_genre_count.show()

In [ ]:
top_genres = valid_genres[:15]

plot_arc_df = genre_arc_df[
    genre_arc_df["primary_genre"].isin(top_genres)
].copy()

plot_arc_df["story_zone"] = pd.Categorical(
    plot_arc_df["story_zone"],
    categories=zone_order,
    ordered=True
)

plot_arc_df = plot_arc_df.sort_values(["primary_genre", "story_zone"])

fig_arc = px.line(
    plot_arc_df,
    x="story_zone",
    y="avg_sentiment",
    color="primary_genre",
    markers=True,
    title="Emotional Arcs Across Movie Genres",
    labels={
        "story_zone": "Story Zone",
        "avg_sentiment": "Average Scene Sentiment",
        "primary_genre": "Genre"
    }
)

fig_arc.add_hline(y=0, line_dash="dash", line_color="gray")
fig_arc.update_traces(line=dict(width=4), marker=dict(size=10))
fig_arc.update_layout(template="plotly_white", width=950, height=600)

fig_arc.show()

In [ ]:
movie_df["opening_to_resolution_change"] = (
    movie_df["resolution_sentiment"] - movie_df["opening_sentiment"]
)

change_df = (
    movie_df
    .groupby("primary_genre", as_index=False)
    .agg(
        avg_change=("opening_to_resolution_change", "mean"),
        num_movies=("movie_id", "nunique")
    )
)

change_df = change_df[change_df["primary_genre"].isin(valid_genres)]
change_df = change_df.sort_values("avg_change", ascending=False)

fig_change = px.bar(
    change_df,
    x="primary_genre",
    y="avg_change",
    title="Average Sentiment Change from Opening to Resolution",
    labels={
        "primary_genre": "Genre",
        "avg_change": "Resolution Sentiment - Opening Sentiment"
    }
)

fig_change.add_hline(y=0, line_dash="dash", line_color="gray")
fig_change.update_layout(
    template="plotly_white",
    xaxis_tickangle=-45
)

fig_change.show()